# Deduplicate candidate records

Normalizes DOI and title fields, flags candidates, and preserves every source row. Automatic flags require manual confirmation.

In [ ]:
from pathlib import Path
import re
import unicodedata
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parents[1]
SOURCE = ROOT / "literature" / "screening.csv"
OUTPUT = ROOT / "literature" / "exports" / "deduplication_candidates.csv"
OUTPUT.parent.mkdir(parents=True, exist_ok=True)

def normalize_doi(value):
    value = "" if pd.isna(value) else str(value).strip().lower()
    value = re.sub(r"^(https?://(dx\\.)?doi\\.org/|doi:\\s*)", "", value)
    return value.rstrip(" .")

def normalize_title(value):
    value = "" if pd.isna(value) else unicodedata.normalize("NFKD", str(value))
    value = "".join(ch for ch in value if not unicodedata.combining(ch)).lower()
    return re.sub(r"[^a-z0-9]+", " ", value).strip()

records = pd.read_csv(SOURCE, dtype=str).fillna("")
records["normalized_doi"] = records["doi"].map(normalize_doi)
records["normalized_title"] = records["title"].map(normalize_title)
records["duplicate_doi"] = records["normalized_doi"].ne("") & records.duplicated("normalized_doi", keep=False)
records["duplicate_title"] = records["normalized_title"].ne("") & records.duplicated("normalized_title", keep=False)
records["duplicate_candidate"] = records["duplicate_doi"] | records["duplicate_title"]
records.to_csv(OUTPUT, index=False)
print(f"Rows: {len(records):,}; duplicate candidates: {records['duplicate_candidate'].sum():,}")
print(f"Review: {OUTPUT}")